###ANALYSIS REPORT 2
--------------------------------------
###Q1. How much of Olist’s total revenue comes from one-time customers, and how much more do repeat customers spend per head? If the repeat rate improved from about 3% to 5%, what additional revenue could the platform expect?

```sql
WITH x AS (SELECT a.customer_unique_id,
       c.price
FROM olist_customers a
     JOIN olist_orders b USING(customer_id)
	 JOIN olist_order_items c USING(order_id)
),
y AS (
    SELECT b.customer_unique_id,
	       CASE WHEN COUNT(DISTINCT a.order_id) > 1 THEN 'repeated'
		   ELSE 'one-time'
		   END AS customer_type
	FROM olist_orders a
	     JOIN olist_customers b
		 USING(customer_id)
	GROUP BY b.customer_unique_id
)
SELECT y.customer_type,
       COUNT(DISTINCT customer_unique_id) AS total_customers,
       SUM(x.price) AS total_revenue,
	   ROUND(SUM(x.price)/COUNT(DISTINCT customer_unique_id), 2) AS revenue_per_customer
FROM x JOIN y USING(customer_unique_id)
GROUP BY y.customer_type
```
OUTPUT :
| customer_type | total_customers | total_revenue | revenue_per_customer |
|---|---:|---:|---:|
| one-time | 92,431 | 12,812,821.73 | 138.62 |
| repeated | 2,989 | 778,821.97 | 260.56 |

>**Insight** : One-time customers account for about 94% of Olist’s revenue (BRL 12.8M of BRL 13.6M), while repeat customers are only about 3% of the customer base. A repeat customer spends about 1.9x more than a one-time customer (BRL 260.56 vs. BRL 138.62). If the repeat rate rose from about 3% to 5%, the platform could earn roughly BRL 217K in additional revenue, an increase of about 1.6%. This is an estimate that assumes new repeat customers would spend like existing ones.



###Q2. Which product categories have the lowest customer repeat rates? Are marketing efforts being wasted on these categories, or are they naturally associated with one-time purchases?

```sql
WITH x AS (
       SELECT b.customer_unique_id,
              ROW_NUMBER() OVER(
                    PARTITION BY b.customer_unique_id
			        ORDER BY a.order_purchase_timestamp
	          ) AS order_no,
	          d.product_category
       FROM olist_orders a
            JOIN olist_customers b USING(customer_id)
	        JOIN olist_order_items c USING(order_id)
	        JOIN olist_products d USING(product_id)
       WHERE a.order_status = 'delivered'
),
y AS (
    SELECT b.customer_unique_id,
	       CASE WHEN COUNT(DISTINCT a.order_id) > 1 THEN 'repeated'
		   ELSE 'one-time'
		   END AS customer_type
	FROM olist_orders a
	     JOIN olist_customers b
		 USING(customer_id)
	GROUP BY b.customer_unique_id  
)
SELECT product_category,
       COUNT(*) AS total_customers,
	   SUM(CASE WHEN customer_type = 'repeated' THEN 1 ELSE 0 END) AS repeated_customers,
	   ROUND(100.0 * SUM(CASE WHEN customer_type = 'repeated' THEN 1 ELSE 0 END)/COUNT(*), 2) AS repeat_pct
FROM x JOIN Y USING(customer_unique_id)
WHERE x.order_no = 1
GROUP BY product_category
HAVING COUNT(*) >= 200
ORDER BY repeat_pct
```

OUTPUT :    
| product_category | total_customers | repeated_customers | repeat_pct |
|---|---:|---:|---:|
| books_technical | 250 | 2 | 0.80 |
| construction_tools_lights | 226 | 3 | 1.33 |
| consoles_games | 995 | 18 | 1.81 |
| electronics | 2462 | 45 | 1.83 |
| cool_stuff | 3465 | 69 | 1.99 |
| musical_instruments | 596 | 12 | 2.01 |
| small_appliances | 592 | 12 | 2.03 |
| luggage_accessories | 987 | 21 | 2.13 |
| kitchen_dining_laundry_garden_furniture | 231 | 5 | 2.16 |
| auto | 3698 | 80 | 2.16 |
| industry_commerce_and_business | 226 | 5 | 2.21 |
| watches_gifts | 5318 | 119 | 2.24 |
| home_appliances_2 | 218 | 5 | 2.29 |
| stationery | 2203 | 52 | 2.36 |
| baby | 2698 | 66 | 2.45 |
| books_general_interest | 485 | 12 | 2.47 |
| toys | 3670 | 91 | 2.48 |
| telephony | 3992 | 99 | 2.48 |
| office_furniture | 1223 | 31 | 2.53 |
| construction_tools_construction | 706 | 18 | 2.55 |
| market_place | 264 | 7 | 2.65 |
| housewares | 5494 | 156 | 2.84 |
| home_construction | 449 | 13 | 2.90 |
| unknown | 1330 | 39 | 2.93 |
| perfumery | 2994 | 88 | 2.94 |
| health_beauty | 8361 | 247 | 2.95 |
| computers_accessories | 6291 | 194 | 3.08 |
| pet_shop | 1630 | 51 | 3.13 |
| garden_tools | 3336 | 105 | 3.15 |
| audio | 340 | 11 | 3.24 |
| food_drink | 212 | 7 | 3.30 |
| food | 423 | 14 | 3.31 |
| fixed_telephony | 206 | 7 | 3.40 |
| home_comfort | 357 | 14 | 3.92 |
| sports_leisure | 7193 | 293 | 4.07 |
| drinks | 267 | 11 | 4.12 |
| bed_bath_table | 8727 | 405 | 4.64 |
| air_conditioning | 235 | 11 | 4.68 |
| furniture_decor | 5914 | 279 | 4.72 |
| furniture_living_room | 385 | 19 | 4.94 |
| fashion_shoes | 228 | 13 | 5.70 |
| fashion_bags_accessories | 1695 | 104 | 6.14 |

>**Insight** : Repeat purchase rates vary by first-order category, from about 1.8% in electronics and consoles_games to about 4.7% in bed_bath_table and furniture_decor, against an overall rate of about 3.1%. Large categories like electronics (2,457 customers, 1.83%) and cool_stuff (3,465 customers, 1.96%) retain customers noticeably worse than home-related categories. Categories with fewer than about 300 customers were treated cautiously, since a few customers can swing the percentage.


###Q3. How many days after their first order do repeat customers place their second order, and after what point does the chance of a return drop sharply?

```sql
WITH abc AS (
       WITH x AS (
             SELECT b.customer_unique_id,
	                ROW_NUMBER() OVER(
                           PARTITION BY b.customer_unique_id
				           ORDER BY a.order_purchase_timestamp
			         ) AS order_no,
			         a.order_purchase_timestamp
	          FROM olist_orders a
	               JOIN olist_customers b
		           USING(customer_id)
       ),
       y AS (
            SELECT b.customer_unique_id,
	               CASE WHEN COUNT(DISTINCT a.order_id) > 1 THEN 'repeated'
		           ELSE 'one-time'
		           END AS customer_type
	         FROM olist_orders a
	              JOIN olist_customers b
		          USING(customer_id)
	         GROUP BY b.customer_unique_id  
       )
       SELECT customer_unique_id,
              MAX(order_purchase_timestamp)::date - MIN(order_purchase_timestamp) ::date AS return_days
        FROM x JOIN y USING(customer_unique_id)
        WHERE x.order_no IN (1,2) AND y.customer_type = 'repeated'
        GROUP BY customer_unique_id
),
bucketed AS (
     SELECT abc.customer_unique_id,
	        abc.return_days,
			CASE WHEN return_days <= 30 THEN '0-30 days'
			WHEN return_days <= 60 THEN '31-60 days'
			WHEN return_days <= 90 THEN '61-90 days'
			ELSE '90+ days'
			END AS return_bucket
	  FROM abc
)
SELECT return_bucket,
       COUNT(*) AS total_count
FROM bucketed
GROUP BY return_bucket
```

OUTPUT :

| return_bucket | total_count |
|---|---:|
| 0-30 days | 1533 |
| 31-60 days | 323 |
| 61-90 days | 203 |
| 90+ days | 938 |

>**Insight** : Among repeat customers, about 51% place their second order within 30 days of their first, but the rate drops sharply afterward (about 11% in days 31-60 and 7% in days 61-90). Around 31% return after more than 90 days. This suggests that the first month after a purchase is the most important window for re-engagement, such as a follow-up offer or reminder, while a smaller group returns after a long gap.

###Q4. Do customers who had a bad first experience (late delivery and a low review) return less often than customers who had a good first experience?

```sql
WITH exp_data AS (
      WITH x AS (
             SELECT b.customer_unique_id,
                    ROW_NUMBER() OVER(
                           PARTITION BY b.customer_unique_id
			               ORDER BY a.order_purchase_timestamp
	                 ) AS order_no,
	                c.review_score,
	                a.delivery_status
             FROM olist_orders a
                  JOIN olist_customers b USING(customer_id)
	              JOIN olist_reviews c USING(order_id)
	         WHERE order_status = 'delivered'
     ),
     y AS (
            SELECT b.customer_unique_id,
	               CASE WHEN COUNT(DISTINCT a.order_id) > 1 THEN 'repeated'
		           ELSE 'one-time'
		           END AS customer_type
	        FROM olist_orders a
	             JOIN olist_customers b
		         USING(customer_id)
	        GROUP BY b.customer_unique_id
    )
    SELECT *
    FROM x JOIN y USING(customer_unique_id)
    WHERE x.order_no = 1
),
bucketed AS (
     SELECT exp_data.customer_unique_id,
	        CASE WHEN exp_data.delivery_status IN ('late','slightly-late') AND exp_data.review_score IN (1,2) THEN 'bad'
			WHEN exp_data.delivery_status IN ('early','on-time') AND exp_data.review_score IN (4,5) THEN 'good'
			ELSE 'moderate'
			END AS experience_group,
			exp_data.customer_type
	 FROM exp_data
)
SELECT experience_group,
       COUNT(*) AS total_customers,
	   ROUND(100.0 * SUM(CASE WHEN customer_type = 'repeated' THEN 1 ELSE 0 END)/COUNT(*), 2) AS repeat_pct
FROM bucketed
GROUP BY experience_group
```

OUTPUT :

| experience_group | total_customers | repeat_pct |
|---|---:|---:|
| bad | 3886 | 2.57 |
| good | 71512 | 3.19 |
| moderate | 17357 | 3.39 |

>**Insight** : Customers whose first order was both late and poorly rated (1-2 stars) returned at a rate of 2.55%, compared with 3.18% for customers with an on-time, highly rated first order. A bad first experience is associated with a lower repeat rate, but the gap is only about 0.6 percentage points, and even satisfied customers rarely return. Poor delivery experience therefore explains only a small part of the very low repurchase rate.